In [6]:
import json

# Load the result dictionary from the JSON file
with open('/Users/kesiyun/Desktop/00workspace/00sequenceresult.json', 'r') as f:
    loaded_result = json.load(f)

#for key, value in loaded_result.items():
    #print(key,': ',value)


In [7]:
#load utilities

import torch
import json
import os
import cv2
import numpy as np
import math
import re


from pathlib import Path
from typing import List

def find_avi_files_by_prefix(prefix: str, videos_dir: str) -> List[str]:
    """
    Return a list of AVI filenames (without .avi) in `videos_dir` that start with the given prefix.

    Parameters
    ----------
    prefix : str
        The starting string to match (e.g., "18A").
    videos_dir : str
        Directory containing .avi files.

    Returns
    -------
    List[str]
        Filenames without the `.avi` extension, sorted alphabetically.
    """
    dir_path = Path(videos_dir)
    if not dir_path.is_dir():
        raise NotADirectoryError(f"Not a directory: {videos_dir}")

    prefix = prefix.strip()
    results = []

    for avi in dir_path.glob("*.avi"):
        name_no_ext = avi.stem  # filename without .avi
        if name_no_ext.startswith(prefix):
            results.append(name_no_ext)

    return sorted(results)




from datetime import datetime, timedelta

def get_video_duration_seconds(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    #print(fps)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    cap.release()
    return duration

def timestamp_to_clip_index(timestamp, length_of_clips):
    # Parse the timestamp string (e.g., "0:03")
    minutes, seconds = map(int, timestamp.split(':'))
    total_seconds = minutes * 60 + seconds

    # Calculate which clip this timestamp falls into
    clip_index = total_seconds // length_of_clips

    return clip_index

# Example usage:
#print(timestamp_to_clip_index("0:03", 4))  # ➜ 0
#print(timestamp_to_clip_index("0:05", 3))  # ➜ 1
#print(timestamp_to_clip_index("0:10", 5))  # ➜ 2


def get_timestamp(length_of_clip, clip_index):
    total_seconds = length_of_clip * clip_index
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"{minutes}:{seconds:02d}"


# Examples:
#print(get_timestamp(5, 0))  # Output: 0:00
#print(get_timestamp(5, 1))  # Output: 0:05
#print(get_timestamp(5, 3))  # Output: 0:15



def find_contractions(list_of_clips):
    C_locations=[]
    for i in range(len(list_of_clips)):
        if 'C' in list_of_clips[i]:
            C_locations.append(i)
    return C_locations



def timestamp_to_seconds_temp(timestamp): #unused but reserved
    minutes, seconds = map(int, timestamp.split(':'))
    return minutes * 60 + seconds

def timestamp_to_seconds(ts):
    """Convert 'M:SS' or 'H:MM:SS' to seconds."""
    parts = list(map(int, ts.split(':')))
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    elif len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    else:
        raise ValueError("Invalid timestamp format")


def extract_pairs(input_string):
    # Example usage
    #input_str = "R P(0:00) A(0:01)C(0:02) P(1:47) A(1:47)C(1:48)"
    #result = extract_pairs(input_str) #[['R', '0:00'], ['P', '0:00'], ['A', '0:01'], ['C', '0:02'], ['P', '1:47'], ['A', '1:47'], ['C', '1:48']]
    # Always start with 'R' at '0:00'
    pairs = [['R', '0:00']]
    
    # Find all matches like A(1:38), C(1:52), etc.
    matches = re.findall(r'([A-Z])\((\d+:\d+)\)', input_string)
    
    # Append each match as a [letter, timestamp] pair
    for letter, timestamp in matches:
        pairs.append([letter, timestamp])
    
    return pairs



def is_timestamp_covered(reference_list, target_timestamp):
    #reference = ['0:00', '2:47', '4:48']
    #print(is_timestamp_covered(reference, '2:48'))  # False
    #print(is_timestamp_covered(reference, '2:43'))  # True
    #print(is_timestamp_covered(reference, '2:42'))  # True
    #print(is_timestamp_covered(reference, '2:41'))  # False
    # Convert target timestamp to datetime object
    target_minutes, target_seconds = map(int, target_timestamp.split(":"))
    target_total_seconds = target_minutes * 60 + target_seconds
    end_total_seconds = target_total_seconds + 5

    for ref in reference_list:
        ref_minutes, ref_seconds = map(int, ref.split(":"))
        ref_total_seconds = ref_minutes * 60 + ref_seconds
        if target_total_seconds <= ref_total_seconds <= end_total_seconds:
            return True
    return False

import time
import sys

def simple_progress_bar(current, total=1):
    if current=='writing':
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        sys.stdout.write(f'\rwriting')
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        return
    if current=='saved':
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        sys.stdout.write(f'\rsaved  ')
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        return
        
    rate = float(current) / total
    arrow = '-' * int(rate * 50) + '>'  # 进度条的视觉表示
    percentage = int(rate * 100)
    # 使用 \r 将光标移到行首，覆盖之前的输出
    sys.stdout.write(f'\r[{arrow:<50}] {percentage}%')
    sys.stdout.flush() # 刷新输出缓冲区，确保立即显示






In [8]:
# specify location of profiles
#REMARK: ONLY PROCESSED UNTIL LAST CLIP (i.e. LAST NOTATIONS IN mmc2.xlsx)

virtual_clip_dict_profile='/Users/kesiyun/Desktop/00workspace/00full_clips_5s.json'#'/content/drive/MyDrive/Colab Notebooks/sten/00virtual_clips_5s.json'
video_split_profile='/Users/kesiyun/Desktop/00workspace/00full_video_split_5s.json'#'/content/drive/MyDrive/Colab Notebooks/sten/00video_split_5s.json'
sequence_result_profile='/Users/kesiyun/Desktop/00workspace/00sequenceresult.json'#'/content/drive/MyDrive/Colab Notebooks/sten/00sequenceresult.json'

In [9]:
#load video and profiles
import json
with open(sequence_result_profile, 'r') as f:
    loaded_sequence = json.load(f)
with open(virtual_clip_dict_profile, 'r') as f:
    loaded_virtual_clip_dict = json.load(f)
with open(video_split_profile, 'r') as f:
    loaded_split_data = json.load(f)

testing_set = loaded_split_data['testing_set']
training_set = loaded_split_data['training_set']

# count Training-Testing configurations
##total
total_clips=0
key_count=0
for key, value in loaded_virtual_clip_dict.items():
    #print(key,": ",value['num_of_clips'] )
    total_clips+=value['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each)\n')

##Testing
print("Testing Set:", testing_set)
total_clips=0
key_count=0
for videos in testing_set:
    total_clips+=loaded_virtual_clip_dict[videos]['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each) \n')

##Training
print("Training Set:", training_set)
total_clips=0
key_count=0
for videos in training_set:
    total_clips+=loaded_virtual_clip_dict[videos]['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each) \n')



8881 clips in 57 videos (average 155.80701754385964 each)

Testing Set: ['16A', '4C', '4A', '4E', '12B', '17I']
1644 clips in 6 videos (average 274.0 each) 

Training Set: ['1A', '1B', '1C', '1D', '1E', '1F', '2A', '3A', '3B', '3C', '3D', '4B', '4D', '4F', '5A', '6A', '6B', '6C', '6D', '7A', '8A', '8B', '9A', '10A', '10B', '10C', '11A', '11B', '11C', '11D', '12A', '13A', '14A', '14B', '14C', '14D', '14E', '15A', '15B', '15C', '15D', '17A', '17B', '17C', '17D', '17E', '17F', '17G', '17H', '18A', '18B']
7237 clips in 51 videos (average 141.90196078431373 each) 



In [10]:
#SlowFast
import torch
from pytorchvideo.models.hub import slowfast_r50
import torchvision.transforms as T
import cv2
import numpy as np

# Load pretrained SlowFast model
#model = slowfast_r50(pretrained=True)
#model.eval()

def load_slowfast_feature_extractor():
    model = slowfast_r50(pretrained=True)
    # Remove the final classification layer to get the 2304-dim feature vector
    model.blocks[-1].proj = torch.nn.Identity()
    return model

# Load the modified model
model = load_slowfast_feature_extractor()
model.eval()

# Video to tensor utility
def video_to_tensor(video_path, num_frames=32, size=224):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        frames.append(frame)
    cap.release()
    # Pad if not enough frames
    while len(frames) < num_frames:
        frames.append(np.zeros_like(frames[0]))
    # Convert to tensor: (C, T, H, W)
    frames = np.stack(frames, axis=0)
    frames = frames.transpose(3, 0, 1, 2)  # (T, H, W, C) -> (C, T, H, W)
    frames = torch.from_numpy(frames).float() / 255.0
    return frames.unsqueeze(0)  # Add batch dimension


def extract_clip_frames(video_path, start_timestamp, length_of_clip, size=224, num_frames=32):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_sec = timestamp_to_seconds(start_timestamp)
    end_sec = start_sec + length_of_clip
    start_frame = int(start_sec * fps)
    end_frame = int(end_sec * fps)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Seek to start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frames = []
    for i in range(start_frame, min(end_frame, total_frames)):
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        frames.append(frame)
    cap.release()

    # Uniformly sample num_frames from the clip
    if len(frames) < num_frames:
        # Pad with black frames if not enough
        while len(frames) < num_frames:
            frames.append(np.zeros_like(frames[0]))
    else:
        idxs = np.linspace(0, len(frames)-1, num_frames).astype(int)
        frames = [frames[i] for i in idxs]

    # Convert to tensor: (C, T, H, W)
    frames = np.stack(frames, axis=0)
    frames = frames.transpose(3, 0, 1, 2)  # (T, H, W, C) -> (C, T, H, W)
    frames = torch.from_numpy(frames).float() / 255.0
    return frames.unsqueeze(0)  # Add batch dimension


def pack_pathway_output(frames, alpha=4):
    """
    Prepare input for SlowFast model.
    frames: torch.Tensor of shape (1, 3, T, H, W)
    alpha: temporal stride between fast and slow pathway (default 4)
    Returns: list of [slow_pathway, fast_pathway]
    """
    fast_pathway = frames
    # Subsample for slow pathway
    slow_pathway = frames[:, :, ::alpha, :, :]
    return [slow_pathway, fast_pathway]

def SlowFast_feature(avi_path, start_timestamp, length_of_clip):
  video_tensor = extract_clip_frames(avi_path, start_timestamp, length_of_clip)
  inputs = pack_pathway_output(video_tensor)  # List of [slow, fast]
  with torch.no_grad():
      features = model(inputs)
  return features



In [20]:
#set video here except for batch
length_of_clip=5
videos_dir='/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug'#'/Volumes/Green SSD/00Workspace_portable/videos_aug/' ####
videos_dir='/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj'
video_name='1A'#training_set[2]#'1A' ####
video_name0=video_name
video_name=video_name+'_0152_C'

#avi_path=videos_dir+'/'+video_name+'.avi'
avi_path = os.path.join(videos_dir, video_name + '.avi')


#print referrence
print('Video file:',avi_path)
print('Sequence:', loaded_sequence[video_name0])
#print('Clips (',length_of_clip,'s):', loaded_virtual_clip_dict[video_name],)
#list_of_contractions=find_contractions(loaded_virtual_clip_dict[video_name]['list_of_clips'])
#print('Contractions in:',list_of_contractions)

Video file: /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj/1A_0152_C.avi
Sequence: {'abstract': 'R P ACD', 'timesec': 'R P(0:39) A(1:38)C(1:52)D(4:25)'}


In [21]:
#SlowFast for a single clip

test_feature_list=[]
label=0

#specified_clip_i=0
#i=specified_clip_i
#start_timestamp = get_timestamp(length_of_clip, i) #'1:50'
start_timestamp='0:05'
features = SlowFast_feature(avi_path, start_timestamp, length_of_clip)

test_feature_list.append([features,label])
print(features.shape)
print(len(test_feature_list))
print(len(test_feature_list[0]))
print(test_feature_list[0][0].shape)
print(test_feature_list[0][1])

torch.Size([1, 2304])
1
2
torch.Size([1, 2304])
0


In [23]:
#SlowFast METHOD for a whole video: into a full list
#print(video_name) #1A (str)

# Example usage
#input_str = "R P(0:00) A(0:01)C(0:02) P(1:47) A(1:47)C(1:48)"
#result = extract_pairs(input_str)
#print(result) #[['R', '0:00'], ['P', '0:00'], ['A', '0:01'], ['C', '0:02'], ['P', '1:47'], ['A', '1:47'], ['C', '1:48']]

def video_feature_extr_SlowFast_shift_window(loaded_virtual_clip_dict,video_name): #argmented 10sec videos
    #need to input processed video_name like '18B_4035_C'
    
    full_feature_list=[]
    this_avi_path = os.path.join(videos_dir, video_name + '.avi') #videos_dir unchanged
    
    #get total seconds of file
    total_seconds = math.ceil(get_video_duration_seconds(this_avi_path))
    #total_clips=math.ceil(total_seconds/5)
    #processed_clips=len(loaded_virtual_clip_dict[video_name]['list_of_clips'])
    #print(processed_clips,'/',total_clips, 'clips processed')

    print('\nExtracting (shift_windows):', this_avi_path)
    print(video_name, 'covered', total_seconds, 'seconds, estimate: ',0.0092*total_seconds,'MB')

    
    is_temp_loaded=True
    temp_count=0
    try:
        temp_features=torch.load(os.path.join(features_dir, video_name + 'temp.pt')) #key?? need to be processed format also.
    except:
        is_temp_loaded=False
        
    if is_temp_loaded:
        temp_count=len(temp_features)
        full_feature_list=temp_features

    #shiftwindow
    for i in range (total_seconds-length_of_clip):
        simple_progress_bar(i, total_seconds-length_of_clip)
        if i<temp_count: continue
        start_timestamp = get_timestamp(1, i) #'1:50'
        torch.save(full_feature_list, os.path.join(features_dir, video_name + 'temp2.pt'))
        feature = SlowFast_feature(this_avi_path, start_timestamp, length_of_clip)
        label=1
        #if is_timestamp_covered(contraction_ts_reference, start_timestamp):
            #label=1
        full_feature_list.append([feature,label])
        simple_progress_bar('writing', 100)
        #time.sleep(0.5) 
        torch.save(full_feature_list, os.path.join(features_dir, video_name + 'temp.pt'))
        simple_progress_bar('saved', 100)

        try: 
            if print_progress:
                print(i)  
        except:
            continue

    return full_feature_list



In [24]:
#BATCH processing for SF features

#REMARK: ONLY PROCESSED UNTIL LAST CLIP (i.e. LAST NOTATIONS IN mmc2.xlsx)


features_dir='/Users/kesiyun/Desktop/00workspace/features/01SF_sw_aug'
features_dir='/Users/kesiyun/Desktop/00workspace/features/01SF_sw_aug_segcrop'
features_dir='/Users/kesiyun/Desktop/00workspace/features/01SF_sw_aug_segobj'



print_progress=False
#print_progress=True

thread_list=[]
for key, value in loaded_virtual_clip_dict.items():
    thread_list.append(key)
#thread_list=thread_list[0:math.ceil(len(thread_list)/2)]

for key, value in loaded_virtual_clip_dict.items():
    #(says, we get strings 1A, 1B, 4A etc. for key)
    if key not in thread_list:
        print(key, 'reserved for another thread')
        continue
        
    contraction_parts=find_avi_files_by_prefix(key, videos_dir)
    for video_name in contraction_parts:
        video_name
        if video_name.endswith("_A"): 
            print(video_name, 'skiped (A)')
            continue
        try: # video might not exist
            if not os.path.exists(os.path.join(features_dir, video_name + '.pt')):#if key.pt does not esist in features_dir, e.g., there is no "1A.pt" in folder "/Users/kesiyun/Desktop/00workspace/features/00SF"
                #video_name=key
                #video_name=key#processed to formated video_name s
                
                full_feature_list_this=video_feature_extr_SlowFast_shift_window(loaded_virtual_clip_dict,video_name)
                torch.save(full_feature_list_this, os.path.join(features_dir, video_name+'.pt'))
                os.remove(os.path.join(features_dir, video_name+'temp.pt'))
                os.remove(os.path.join(features_dir, video_name+'temp2.pt'))
                print(video_name, 'processed')
                
            else: # else key.pt (says 1A.pt) already exists
                print(video_name, 'skiped')
                continue
        except Exception as e:
            print(video_name, 'not processed (video might not exist now):', e)

1A_0152_A skiped (A)

Extracting (shift_windows): /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj/1A_0152_B.avi
1A_0152_B covered 10 seconds, estimate:  0.092 MB
saved  1A_0152_B processed--------------->         ] 80%

Extracting (shift_windows): /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj/1A_0152_C.avi
1A_0152_C covered 10 seconds, estimate:  0.092 MB
saved  1A_0152_C processed--------------->         ] 80%

Extracting (shift_windows): /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj/1A_0152_D.avi
1A_0152_D covered 10 seconds, estimate:  0.092 MB
saved  1A_0152_D processed--------------->         ] 80%

Extracting (shift_windows): /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj/1A_0152_E.avi
1A_0152_E covered 10 seconds, estimate:  0.092 MB
saved  1A_0152_E processed--------------->         ] 80%

Extracting (shift_windows): /Volum

In [12]:

import os
print(os.cpu_count())


4


In [11]:
#Experimental script to read(check) the contents
feature_list_load_1=torch.load('/Users/kesiyun/Desktop/00workspace/features/01SF_sw_aug/18B_4035_B.pt')
print('Count:',len(feature_list_load_1))

index=-1
print('Feature:', feature_list_load_1[index][0].shape)
print('Label:', feature_list_load_1[index][1])

print('Label-1s:', feature_list_load_1[index][1])
print(feature_list_load_1)

feature_list_load_1=torch.load('/Users/kesiyun/Desktop/00workspace/features/01SF_sw_aug/18B_4035_C.pt')
print('Count:',len(feature_list_load_1))

index=-1
print('Feature:', feature_list_load_1[index][0].shape)
print('Label:', feature_list_load_1[index][1])

print('Label-1s:', feature_list_load_1[index][1])
print(feature_list_load_1)



Count: 5
Feature: torch.Size([1, 2304])
Label: 1
Label-1s: 1
[[tensor([[0.0074, 0.3289, 0.0430,  ..., 0.0272, 0.2466, 0.7281]]), 1], [tensor([[0.0022, 0.3193, 0.0839,  ..., 0.0081, 0.2831, 0.7184]]), 1], [tensor([[0.0081, 0.3121, 0.0569,  ..., 0.0143, 0.2238, 0.9923]]), 1], [tensor([[0.0144, 0.3527, 0.1719,  ..., 0.0155, 0.1817, 0.7568]]), 1], [tensor([[0.0190, 0.3735, 0.0845,  ..., 0.0191, 0.1755, 0.8726]]), 1]]
Count: 5
Feature: torch.Size([1, 2304])
Label: 1
Label-1s: 1
[[tensor([[0.1002, 0.1136, 0.1338,  ..., 0.1443, 0.2745, 0.5603]]), 1], [tensor([[0.2647, 0.0566, 0.2498,  ..., 0.1557, 0.2985, 0.6462]]), 1], [tensor([[0.1555, 0.0950, 0.1614,  ..., 0.2613, 0.2780, 0.9341]]), 1], [tensor([[0.2001, 0.1548, 0.3090,  ..., 0.0968, 0.2347, 0.6714]]), 1], [tensor([[0.3030, 0.2000, 0.1665,  ..., 0.1343, 0.2281, 0.8393]]), 1]]


In [12]:

videos_dir = '/Volumes/Green SSD/00Workspace_portable/videos_aug/'
contraction_parts=find_avi_files_by_prefix("18A", videos_dir)
print(contraction_parts)
# Output: ['18A_3922_B', '18A_3922_C', '18A_3713_B']  # if those exist


['18A_0120_A', '18A_0120_B', '18A_0120_C', '18A_0120_D', '18A_0120_E', '18A_0120_F', '18A_0120_G', '18A_0120_H', '18A_0507_A', '18A_0507_B', '18A_0507_C', '18A_0507_D', '18A_0507_E', '18A_0507_F', '18A_0507_G', '18A_0507_H', '18A_1344_A', '18A_1344_B', '18A_1344_C', '18A_1344_D', '18A_1344_E', '18A_1344_F', '18A_1344_G', '18A_1344_H']


In [40]:
video_name="1C_0036_A"

print(video_name.endswith("_A"))

True


In [ ]:
prefix = '18A'
print(prefix)
prefix = prefix.strip()
print(prefix)
